In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("whitegrid")

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.preprocessing import PolynomialFeatures


from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import BayesianRidge

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import confusion_matrix

In [2]:
np.random.uniform(30, 180, 10).round()

array([138., 106.,  76.,  71.,  89., 132., 110., 162., 121.,  54.])

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. Параметры генерации
# ============================================
N = 30 # Чем меньше N, тем сильнее будет разброс OLS
np.random.seed(42)

# ============================================
# 2. Генерация признаков
# ============================================

data = {}

data['AdBudget'] = np.random.uniform(100, 200, N).round(2)
data['ChannelsCount'] = np.random.choice([1, 2, 3, 4, 5], N, p=[0.05, 0.1, 0.4, 0.3, 0.15])
data['LeadScoreThreshold'] = np.random.randint(20, 80, N)
data['EmailFollowUps'] = np.random.randint(0, 10, N)
data['DemoRequests'] = data['EmailFollowUps'] + np.random.choice([0, 1], N, p=[0.9, 0.1])
data['AvgTimeOnSite'] = np.random.uniform(30, 180, N).round(2)
data['PagesPerSession'] =(data['AvgTimeOnSite']/15 + np.random.normal(0, 0.01, N)).round(2)
data['ContentPieces'] = np.random.choice([1, 2, 3, 4, 5], N, p=[0.6, 0.2, 0.1, 0.05, 0.05])
data['TestRatio'] = np.random.uniform(0, 1, N).round(4)
data['RetargetingShare'] = np.random.uniform(0, 1, N).round(4)
data['OrganicShare'] = np.random.uniform(0.1, 0.3, N).round(4)
data['CAC'] = np.random.randint(20, 80, N)
data['LeadVolume'] = np.random.randint(20, 80, N)
data['SaleCycleDays'] = np.random.uniform(1, 3, N).round(2)
data['WeekendPosts'] = np.random.randint(100, 150, N)

                                    
df = pd.DataFrame(data)
df_scaled = df
df_scaled = (df - df.min())/(df.max() - df.min())

true_coeffs = {
    'AdBudget': 5,
    
    'ChannelsCount': 5,
    
    'LeadScoreThreshold': -20,
    
    'EmailFollowUps': 20,
    
    'DemoRequests': 5,
    
    'AvgTimeOnSite': 1,
    
    'PagesPerSession': 1,
    
    'ContentPieces': 5,
    
    'TestRatio': 20,
    
    'RetargetingShare': 4,
    
    'OrganicShare': 0.1,
    
    'CAC': 20,
    
    'LeadVolume': 1,
    
    'SaleCycleDays': 0.1,
    
    'WeekendPosts': 0.1,
}

X = df_scaled[list(true_coeffs.keys())].values
true_coef_array = np.array(list(true_coeffs.values()))
y_linear = X @ true_coef_array
logits = y_linear + np.random.normal(0, 3, N)
y_proba = logits
y_proba = (100*(y_proba - y_proba.min())/(y_proba.max() - y_proba.min())).astype(int)

#######################################
# Target = ConversionRate 
#######################################

df['ConversionRate'] = y_proba

print(f"Датасет готов: {df.shape[0]} строк, {len(true_coeffs)} признаков + таргет")
print(f"Таргет: от {y_proba.min():.3f} до {y_proba.max():.3f}, среднее = {y_proba.mean():.3f}")
print('')

display(df)

# df.to_csv('data/marketing.csv', index=False)

X = df.drop(columns = ['Target'])
y = df['Target']

max_r2_test_ridge = 0
max_r2_test_lasso = 0

max_alpha_ridge = 0
max_alpha_lasso = 0

train_lr = []
test_lr = []

train_ridge = []
test_ridge = []

train_lasso = []
test_lasso = []

a_h = np.arange(0, 1.5, 0.1)

for alpha in a_h:

    r2_train_lr = []
    r2_test_lr =[]

    r2_train_ridge = []
    r2_test_ridge =[]

    r2_train_lasso = []
    r2_test_lasso =[]

    for k in range(100):

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=k)
        
        scaler = MinMaxScaler()
        scaler.fit(X_train)

        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

        lr = LinearRegression()
        lr.fit(X_train, y_train)

        y_train_pred_lr = lr.predict(X_train)
        y_test_pred_lr = lr.predict(X_test)

        r2_train_lr.append(round(r2_score(y_train, y_train_pred_lr), 4))
        r2_test_lr.append(round(r2_score(y_test, y_test_pred_lr), 4))

        ridge = Ridge(alpha)
        ridge.fit(X_train, y_train)

        y_train_pred_ridge = ridge.predict(X_train)
        y_test_pred_ridge = ridge.predict(X_test)

        r2_train_ridge.append(round(r2_score(y_train, y_train_pred_ridge), 4))
        r2_test_ridge.append(round(r2_score(y_test, y_test_pred_ridge), 4))

        lasso = Lasso(alpha)
        lasso.fit(X_train, y_train)

        y_train_pred_lasso = lasso.predict(X_train)
        y_test_pred_lasso = lasso.predict(X_test)

        r2_train_lasso.append(round(r2_score(y_train, y_train_pred_lasso), 4))
        r2_test_lasso.append(round(r2_score(y_test, y_test_pred_lasso), 4))

    train_lr.append(round(np.array(r2_train_lr).mean(), 4))
    test_lr.append(round(np.array(r2_test_lr).mean(), 4))
    
    train_ridge.append(round(np.array(r2_train_ridge).mean(), 4))
    test_ridge.append(round(np.array(r2_test_ridge).mean(), 4))
    
    train_lasso.append(round(np.array(r2_train_lasso).mean(), 4))
    test_lasso.append(round(np.array(r2_test_lasso).mean(), 4))
    
    if round(np.array(r2_test_ridge).mean(), 4) > max_r2_test_ridge:
        max_r2_test_ridge = round(np.array(r2_test_ridge).mean(), 4)
        max_alpha_ridge = alpha
        
    if round(np.array(r2_test_lasso).mean(), 4) > max_r2_test_lasso:
        max_r2_test_lasso = round(np.array(r2_test_lasso).mean(), 4)
        max_alpha_lasso = alpha
        
    print(round(alpha, 2))
    
print('')
print('Линейная регрессия')
print('r2_train_lr =', round(np.array(train_lr).mean(), 4))
print('r2_test_lr =', round(np.array(test_lr).mean(), 4))
print('')

print('Ридж')
print('max_r2_test_ridge =', max_r2_test_ridge)
print('max_alpha_ridge =', max_alpha_ridge)
print('')

print('Лассо')
print('max_r2_test_lasso =', max_r2_test_lasso)
print('max_alpha_lasso =', max_alpha_lasso)

#################
# Линейная регрессия
#################

X = df.drop(columns = ['Target'])
y = df['Target']

coef_lr = pd.DataFrame(columns = X.columns)

for k in range(100):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=k)
    
    scaler = MinMaxScaler()
    scaler.fit(X_train)

    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    
#     scaler = MinMaxScaler()
#     scaler.fit(y_train.reshape(-1, 1))

#     y_train = scaler.transform(y_train.reshape(-1, 1))
#     y_test = scaler.transform(y_test.reshape(-1, 1))
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    y_train_pred_lr = lr.predict(X_train)
    y_test_pred_lr = lr.predict(X_test)

    coef_lr.loc[k] = lr.coef_

coef_long_lr = pd.DataFrame(columns = ['model', 'feature', 'coef'])
for col in coef_lr.columns:
    d = pd.DataFrame(columns = ['model', 'feature', 'coef'])
    d['model'] = ['OLS']*len(coef_lr)
    d['feature'] = [col]*len(coef_lr)
    d['coef'] = coef_lr[col]
    coef_long_lr = pd.concat([coef_long_lr, d], ignore_index=True)
    
#################
# Ридж
#################

X = df.drop(columns = ['Target'])
y = df['Target']

coef_ridge = pd.DataFrame(columns = X.columns)

for k in range(100):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=k)
    
    scaler = MinMaxScaler()
    scaler.fit(X_train)

    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    
#     scaler = MinMaxScaler()
#     scaler.fit(y_train.reshape(-1, 1))

#     y_train = scaler.transform(y_train.reshape(-1, 1))
#     y_test = scaler.transform(y_test.reshape(-1, 1))
    
    ridge = Ridge(max_alpha_ridge)
    ridge.fit(X_train, y_train)

    y_train_pred_ridge = ridge.predict(X_train)
    y_test_pred_ridge = ridge.predict(X_test)

    coef_ridge.loc[k] = ridge.coef_


coef_long_ridge = pd.DataFrame(columns = ['model', 'feature', 'coef'])
for col in coef_ridge.columns:
    d = pd.DataFrame(columns = ['model', 'feature', 'coef'])
    d['model'] = ['Ridge']*len(coef_ridge)
    d['feature'] = [col]*len(coef_ridge)
    d['coef'] = coef_ridge[col]
    coef_long_ridge = pd.concat([coef_long_ridge, d], ignore_index=True)
    
#################
# Лассо
#################
    
X = df.drop(columns = ['Target'])
y = df['Target']

coef_lasso = pd.DataFrame(columns = X.columns)

for k in range(100):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=k)
    
    scaler = MinMaxScaler()
    scaler.fit(X_train)

    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    
#     scaler = MinMaxScaler()
#     scaler.fit(y_train.reshape(-1, 1))

#     y_train = scaler.transform(y_train.reshape(-1, 1))
#     y_test = scaler.transform(y_test.reshape(-1, 1))
    
    lasso = Lasso(max_alpha_lasso)
    lasso.fit(X_train, y_train)

    y_train_pred_lasso = lasso.predict(X_train)
    y_test_pred_lasso = lasso.predict(X_test)

    coef_lasso.loc[k] = lasso.coef_


coef_long_lasso = pd.DataFrame(columns = ['model', 'feature', 'coef'])
for col in coef_lasso.columns:
    d = pd.DataFrame(columns = ['model', 'feature', 'coef'])
    d['model'] = ['Lasso']*len(coef_lasso)
    d['feature'] = [col]*len(coef_lasso)
    d['coef'] = coef_lasso[col]
    coef_long_lasso = pd.concat([coef_long_lasso, d], ignore_index=True)


coef_long = pd.concat([coef_long_lr, coef_long_ridge, coef_long_lasso], ignore_index=True)

plt.figure(figsize=(20, 5))
sns.boxplot(data=coef_long, x='feature', y='coef', hue = 'model')
plt.xticks(rotation=90)
plt.show()

Датасет готов: 30 строк, 15 признаков + таргет
Таргет: от 0.000 до 100.000, среднее = 60.467



,AdBudget,ChannelsCount,LeadScoreThreshold,EmailFollowUps,DemoRequests,AvgTimeOnSite,PagesPerSession,ContentPieces,TestRatio,RetargetingShare,OrganicShare,CAC,LeadVolume,SaleCycleDays,WeekendPosts,ConversionRate
0,137.45,4,35,2,2,82.89,5.52,1,0.7442,0.9803,0.1794,75,41,1.93,149,77
1,195.07,3,64,0,0,75.72,5.04,5,0.7209,0.0753,0.1102,38,48,2.30,134,23
2,173.20,2,37,4,4,54.70,3.65,5,0.3081,0.3057,0.2773,47,74,1.10,122,60
3,159.87,5,66,9,9,110.11,7.33,1,0.5425,0.1909,0.1055,77,22,2.90,116,89
4,115.60,5,72,6,6,102.72,6.85,3,0.5088,0.2685,0.2158,74,31,2.77,125,55
5,115.60,4,43,9,9,133.87,8.92,1,0.6363,0.4853,0.1877,45,45,1.52,107,84
6,105.81,3,45,8,8,70.41,4.71,1,0.2505,0.3727,0.2344,56,35,1.03,128,66
7,186.62,2,44,6,6,66.62,4.44,2,0.5899,0.3947,0.1656,45,70,2.87,125,71
8,160.11,4,79,8,8,55.24,3.68,3,0.9789,0.8442,0.1310,72,56,2.00,109,92
9,170.81,3,79,7,7,62.81,4.19,3,0.4867,0.9300,0.2964,42,41,2.08,125,39


KeyError: "['Target'] not found in axis"

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 2.5), sharey=True)

ax[0].plot(a_h, train_ridge, color = 'C0', label = 'train')
ax[0].plot(a_h, test_ridge, color = 'C1', label = 'test')
ax[0].set_title('Ридж')
ax[0].legend()

ax[1].plot(a_h, train_lasso, color = 'C0', label = 'train')
ax[1].plot(a_h, test_lasso, color = 'C1', label = 'test')
ax[1].set_title('Лассо')
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
mask = coef_long['model'] == 'OLS'
# mask = coef_long['model'] == 'Ridge'
# mask = coef_long['model'] == 'Lasso'
plt.figure(figsize=(20, 5))
sns.boxplot(data=coef_long.loc[~mask], x='feature', y='coef', hue = 'model')
plt.xticks(rotation=90)
plt.show()

In [ ]:
coef_long_lasso

In [ ]:
# df.to_csv('data/hr.csv', index=False)